# Re-ranking in Production RAG Systems
## A Comprehensive Guide to Post-Retrieval Candidate Re-ranking

---

**Scope**: This notebook covers the theory, mathematics, implementation, and production trade-offs of re-ranking methods used after candidate generation in Retrieval-Augmented Generation (RAG) pipelines.

**Audience**: ML Engineers and Applied Scientists building production search/RAG systems.

**Structure**:
1. Why Re-ranking Matters
2. The Retrieve-then-Rerank Paradigm
3. Cross-Encoder Re-ranking
4. Bi-Encoder (Dense Retrieval) Comparison
5. ColBERT — Late Interaction
6. MonoT5 — Seq2Seq Re-ranking
7. LLM-based Listwise Re-ranking
8. Reciprocal Rank Fusion (RRF)
9. Maximal Marginal Relevance (MMR)
10. Learning to Rank (LambdaMART)
11. Production Comparison & Guidelines

## 1. Why Re-ranking Matters in RAG

### The Fundamental Problem

In a RAG pipeline, the **retriever** (sparse or dense) fetches $$k$$ candidate documents from a corpus of $$N$$ documents where $$N \gg k$$. The retriever must be **fast** (sub-linear in $$N$$), which forces it to use approximate representations:

- **Sparse retrieval** (BM25): Bag-of-words, no semantic understanding
- **Dense retrieval** (Bi-Encoder): Independent embeddings, no cross-attention between query and document

This speed–accuracy trade-off means the top-$$k$$ candidates are **not optimally ranked**. A re-ranker applies a more expensive but more accurate scoring function to these $$k$$ candidates.

### The Information Retrieval Cascade

$$
\underbrace{N \text{ documents}}_{\text{Full Corpus}} \xrightarrow{\text{Retriever (fast, O(log N))}} \underbrace{k \text{ candidates}}_{k \approx 100\text{-}1000} \xrightarrow{\text{Re-ranker (slow, O(k))}} \underbrace{m \text{ final results}}_{m \approx 3\text{-}10}
$$

### Why Not Just Use a Better Retriever?

| Concern | Retriever Only | Retrieve + Re-rank |
| --- | --- | --- |
| Latency at corpus scale | Acceptable | Acceptable |
| Semantic accuracy | Moderate | High |
| Cross-attention (query↔doc) | No | Yes |
| Handles lexical mismatch | Poorly (sparse) | Well |
| Production cost | Lower | Slightly higher |

### Quantitative Impact

In practice, adding a cross-encoder re-ranker to a BM25 retriever improves:
- **MRR@10**: +15–25% on MS MARCO
- **NDCG@10**: +10–20% on BEIR benchmarks
- **Answer accuracy in RAG**: +8–15% on downstream QA tasks

The re-ranker is often the **single highest-ROI component** in a production RAG system.

## 2. The Retrieve-then-Rerank Paradigm

### Formal Definition

Given:
- A query $$q$$
- A corpus $$\mathcal{C} = \{d_1, d_2, \ldots, d_N\}$$
- A retriever $$f_{\text{ret}}: (q, \mathcal{C}) \rightarrow \{d_{\pi(1)}, \ldots, d_{\pi(k)}\}$$ that returns top-$$k$$ candidates
- A re-ranker $$f_{\text{rerank}}: (q, d_i) \rightarrow s_i \in \mathbb{R}$$ that scores each candidate

The final ranking is:

$$
\text{rank}(d_i) = |\{d_j : f_{\text{rerank}}(q, d_j) > f_{\text{rerank}}(q, d_i)\}| + 1
$$

### Key Design Principles

**1. The re-ranker sees the full (query, document) pair jointly**

Unlike retrievers that encode query and document independently:

$$
\text{Retriever: } s(q, d) = \phi(q)^T \psi(d) \quad \text{(decomposable)}
$$

$$
\text{Re-ranker: } s(q, d) = f([q; d]) \quad \text{(non-decomposable, joint encoding)}
$$

**2. Computational budget allocation**

The total latency budget $$T$$ is split:

$$
T = T_{\text{retrieval}} + T_{\text{rerank}} = O(\log N) + O(k \cdot c)
$$

where $$c$$ is the per-candidate re-ranking cost. In production, $$k$$ is tuned so that $$T_{\text{rerank}} < T_{\text{SLA}} - T_{\text{retrieval}}$$.

**3. Recall vs. Precision trade-off**

- **Retriever** optimizes for **recall@k**: don't miss relevant documents
- **Re-ranker** optimizes for **precision@m**: put the best documents at the top

This separation of concerns is the key architectural insight.

In [0]:
%pip install sentence-transformers torch numpy scikit-learn lightgbm --quiet
dbutils.library.restartPython()

In [0]:
import torch
import torch.nn.functional as F
import numpy as np
from typing import List, Tuple, Dict

# ============================================================
# SAMPLE DATA: Simulating a RAG retrieval scenario
# ============================================================
# Query: User question for a RAG system
query = "What are the benefits of transformer architecture over RNNs?"

# Candidate documents retrieved by the first-stage retriever (e.g., BM25 or dense retrieval)
candidates = [
    "The transformer architecture, introduced in 'Attention is All You Need' (2017), "
    "eliminates recurrence entirely, relying on self-attention mechanisms. This allows "
    "parallel processing of all positions in a sequence, dramatically reducing training time. "
    "Unlike RNNs, transformers do not suffer from vanishing gradients over long sequences.",
    
    "Recurrent Neural Networks (RNNs) process sequences one token at a time, maintaining "
    "a hidden state. This sequential nature makes them slow to train and prone to forgetting "
    "long-range dependencies despite LSTM and GRU improvements.",
    
    "BERT is a transformer-based model that uses bidirectional self-attention. It achieves "
    "state-of-the-art results on many NLP benchmarks by pre-training on masked language modeling.",
    
    "Convolutional Neural Networks (CNNs) are primarily used for image processing tasks. "
    "They use filters to detect spatial patterns and have been extended to NLP with models "
    "like TextCNN for text classification.",
    
    "The key advantages of transformers include: (1) parallelizable computation enabling "
    "efficient GPU utilization, (2) direct attention between any two positions enabling "
    "better long-range dependency modeling, (3) more interpretable attention weights, "
    "and (4) better scalability to larger model sizes compared to recurrent architectures.",
    
    "GPT models use the transformer decoder architecture with causal (unidirectional) "
    "self-attention. They are trained autoregressively to predict the next token.",
    
    "The attention mechanism computes a weighted sum of values where weights are determined "
    "by the compatibility between queries and keys. Multi-head attention allows the model "
    "to attend to information from different representation subspaces.",
    
    "Python is a popular programming language for machine learning due to its extensive "
    "library ecosystem including PyTorch, TensorFlow, and scikit-learn.",
]

# Ground truth: documents 0, 4 are highly relevant; 1, 6 are somewhat relevant
relevance_labels = [3, 2, 1, 0, 3, 1, 2, 0]  # 0=irrelevant, 3=highly relevant

print(f"Query: {query}")
print(f"Number of candidates: {len(candidates)}")
print(f"\nRelevance distribution: {dict(zip(range(len(candidates)), relevance_labels))}")

## 3. Cross-Encoder Re-ranking

### Core Idea

A **Cross-Encoder** concatenates the query and document into a single input sequence and passes it through a transformer, allowing **full cross-attention** between all query and document tokens.

### Architecture

$$
\text{Input} = [\text{CLS}] \; q_1 \; q_2 \; \ldots \; q_m \; [\text{SEP}] \; d_1 \; d_2 \; \ldots \; d_n \; [\text{SEP}]
$$

$$
h = \text{BERT}([\text{CLS}]; q; [\text{SEP}]; d; [\text{SEP}])
$$

$$
s(q, d) = W^T h_{[\text{CLS}]} + b \quad \text{where } W \in \mathbb{R}^{H}, b \in \mathbb{R}
$$

The $$[\text{CLS}]$$ token representation $$h_{[\text{CLS}]} \in \mathbb{R}^H$$ encodes the joint semantics of the query-document pair.

### Training Objective

Trained with **binary cross-entropy** or **contrastive loss**:

$$
\mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log \sigma(s_i) + (1 - y_i) \log(1 - \sigma(s_i)) \right]
$$

where $$y_i \in \{0, 1\}$$ is the relevance label and $$\sigma$$ is the sigmoid function.

For **listwise training** with multiple relevance grades:

$$
\mathcal{L}_{\text{listwise}} = -\sum_{i=1}^{k} \frac{2^{y_i} - 1}{\text{maxDCG}} \log \frac{e^{s_i}}{\sum_{j=1}^{k} e^{s_j}}
$$

### Self-Attention Complexity

For a query of length $$m$$ and document of length $$n$$, the self-attention over the concatenated sequence has complexity:

$$
O((m + n)^2 \cdot H)
$$

This is why cross-encoders are expensive — every query token attends to every document token and vice versa.

### Pros
- **Highest accuracy**: Full cross-attention captures fine-grained query-document interactions
- **Handles semantic nuance**: Can resolve lexical ambiguity, negation, and complex reasoning
- **Simple to train**: Standard classification head on pre-trained transformers
- **Well-studied**: Extensive benchmarks (MS MARCO, TREC, BEIR)

### Cons
- **Slow inference**: $$O(k)$$ forward passes, each $$O((m+n)^2)$$
- **Cannot pre-compute**: Document representations depend on the query
- **Memory intensive**: Full transformer inference per (query, document) pair
- **Latency**: ~50–200ms for 100 candidates on GPU; much slower on CPU
- **Max sequence length**: Typically 512 tokens, truncating long documents

In [0]:
from sentence_transformers import CrossEncoder
import time

# ============================================================
# METHOD 1: CROSS-ENCODER RE-RANKING
# Using a pre-trained cross-encoder model
# ============================================================

# Load a pre-trained cross-encoder (trained on MS MARCO passage ranking)
print("Loading Cross-Encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)

# Score all (query, candidate) pairs
start_time = time.time()
pairs = [[query, doc] for doc in candidates]
cross_encoder_scores = cross_encoder.predict(pairs)
ce_latency = time.time() - start_time

# Rank candidates by score
ce_ranking = np.argsort(cross_encoder_scores)[::-1]

print(f"\n{'='*70}")
print(f"CROSS-ENCODER RE-RANKING RESULTS")
print(f"{'='*70}")
print(f"Model: cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"Latency: {ce_latency*1000:.1f}ms for {len(candidates)} candidates")
print(f"Per-candidate: {ce_latency*1000/len(candidates):.1f}ms")
print(f"\n{'Rank':<6}{'Score':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, idx in enumerate(ce_ranking, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{cross_encoder_scores[idx]:<12.4f}{relevance_labels[idx]:<12}{preview}")

In [0]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

# ============================================================
# CROSS-ENCODER FROM SCRATCH (PyTorch)
# Understanding the internals
# ============================================================

class CrossEncoderFromScratch(nn.Module):
    """
    A cross-encoder that concatenates query and document,
    passes through BERT, and applies a classification head
    to the [CLS] token representation.
    
    Architecture:
        Input: [CLS] query [SEP] document [SEP]
        Output: scalar relevance score
    """
    
    def __init__(self, model_name: str = "bert-base-uncased", num_labels: int = 1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, 
                token_type_ids: torch.Tensor = None) -> torch.Tensor:
        """
        Forward pass:
        1. Encode concatenated [query; document] through BERT
        2. Extract [CLS] token representation (index 0)
        3. Apply dropout + linear layer for scoring
        """
        # Full transformer encoding with cross-attention between q and d
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        # [CLS] token captures joint query-document semantics
        # Shape: (batch_size, hidden_size)
        cls_representation = outputs.last_hidden_state[:, 0, :]
        
        # Classification head: project to scalar score
        cls_representation = self.dropout(cls_representation)
        logits = self.classifier(cls_representation)  # (batch_size, 1)
        
        return logits.squeeze(-1)
    
    def score_pairs(self, query: str, documents: List[str], 
                    tokenizer, max_length: int = 512) -> np.ndarray:
        """Score a batch of (query, document) pairs."""
        self.eval()
        
        # Tokenize all pairs jointly (this is the key: query and doc see each other)
        encodings = tokenizer(
            [query] * len(documents),  # Repeat query for each document
            documents,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            scores = self.forward(
                input_ids=encodings["input_ids"],
                attention_mask=encodings["attention_mask"],
                token_type_ids=encodings.get("token_type_ids")
            )
        
        return scores.numpy()


# Demonstrate the architecture
print("Cross-Encoder Architecture (from scratch):")
print("="*60)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = CrossEncoderFromScratch("bert-base-uncased")

# Show tokenization of a (query, document) pair
sample_encoding = tokenizer(query, candidates[0], truncation=True, max_length=512, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(sample_encoding["input_ids"][0])

print(f"\nInput structure (first 30 tokens):")
print(f"  {tokens[:30]}")
print(f"\nToken type IDs (0=query, 1=document):")
print(f"  {sample_encoding['token_type_ids'][0][:30].tolist()}")
print(f"\nTotal input length: {len(tokens)} tokens")
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  - Encoder: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  - Classifier head: {sum(p.numel() for p in model.classifier.parameters()):,}")

# Score with untrained model (for demonstration)
scores = model.score_pairs(query, candidates[:3], tokenizer)
print(f"\nScores (untrained, for structure demo): {scores}")

del model, tokenizer  # Free memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Bi-Encoder (Dense Retrieval) — The Baseline to Beat

### Core Idea

A **Bi-Encoder** encodes query and document **independently** into dense vectors, then scores via dot product or cosine similarity. This is what most vector databases use for initial retrieval.

### Architecture

$$
e_q = \text{Encoder}_Q(q) \in \mathbb{R}^d
$$

$$
e_d = \text{Encoder}_D(d) \in \mathbb{R}^d
$$

$$
s(q, d) = e_q^T e_d \quad \text{or} \quad s(q, d) = \cos(e_q, e_d) = \frac{e_q^T e_d}{\|e_q\| \cdot \|e_d\|}
$$

### Why It's Faster but Less Accurate

The score function is **decomposable** — document embeddings can be pre-computed:

$$
\text{Inference cost} = \underbrace{O(m^2 \cdot H)}_{\text{encode query}} + \underbrace{O(k \cdot d)}_{\text{dot products}}
$$

Compare to Cross-Encoder: $$O(k \cdot (m+n)^2 \cdot H)$$

### The Representation Bottleneck

The bi-encoder must compress all document semantics into a single vector $$e_d \in \mathbb{R}^d$$ (typically $$d = 768$$). This creates an **information bottleneck**:

$$
I(q; d) \geq I(q; e_d) \quad \text{(Data Processing Inequality)}
$$

The cross-encoder does not have this bottleneck since it processes the raw tokens jointly.

### Training: Contrastive Learning

$$
\mathcal{L}_{\text{InfoNCE}} = -\log \frac{e^{\text{sim}(q, d^+)/\tau}}{e^{\text{sim}(q, d^+)/\tau} + \sum_{j=1}^{N^-} e^{\text{sim}(q, d_j^-)/\tau}}
$$

where $$\tau$$ is a temperature parameter, $$d^+$$ is the positive document, and $$d_j^-$$ are negatives.

### Pros
- **Extremely fast inference**: Pre-computed embeddings, ANN search in $$O(\log N)$$
- **Scalable**: Works with billions of documents via FAISS/ScaNN
- **Simple deployment**: Embeddings stored in vector databases
- **Real-time**: <10ms for retrieval

### Cons
- **Information bottleneck**: Single vector cannot capture all document semantics
- **No cross-attention**: Cannot model fine-grained query-document interactions
- **Worse on nuanced queries**: Struggles with negation, specific conditions, multi-hop reasoning
- **Embedding space quality**: Heavily dependent on training data distribution

In [0]:
from sentence_transformers import SentenceTransformer

# ============================================================
# METHOD 2: BI-ENCODER (Dense Retrieval)
# For comparison with cross-encoder
# ============================================================

print("Loading Bi-Encoder model...")
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode query and documents INDEPENDENTLY
start_time = time.time()
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)
doc_embeddings = bi_encoder.encode(candidates, convert_to_tensor=True)

# Score via cosine similarity
bi_encoder_scores = F.cosine_similarity(
    query_embedding.unsqueeze(0),  # (1, d)
    doc_embeddings,                 # (k, d)
    dim=1
).cpu().numpy()
be_latency = time.time() - start_time

# Rank
be_ranking = np.argsort(bi_encoder_scores)[::-1]

print(f"\n{'='*70}")
print(f"BI-ENCODER RESULTS (for comparison)")
print(f"{'='*70}")
print(f"Model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {query_embedding.shape[0]}")
print(f"Latency: {be_latency*1000:.1f}ms for {len(candidates)} candidates")
print(f"\n{'Rank':<6}{'Score':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, idx in enumerate(be_ranking, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{bi_encoder_scores[idx]:<12.4f}{relevance_labels[idx]:<12}{preview}")

# ============================================================
# COMPARISON: Cross-Encoder vs Bi-Encoder
# ============================================================
print(f"\n\n{'='*70}")
print(f"COMPARISON: Cross-Encoder vs Bi-Encoder")
print(f"{'='*70}")
print(f"{'Metric':<30}{'Cross-Encoder':<20}{'Bi-Encoder':<20}")
print(f"{'-'*70}")
print(f"{'Latency (total)':<30}{ce_latency*1000:<20.1f}{be_latency*1000:<20.1f}")
print(f"{'Top-1 Relevance':<30}{relevance_labels[ce_ranking[0]]:<20}{relevance_labels[be_ranking[0]]:<20}")
print(f"{'Top-3 Avg Relevance':<30}"
      f"{np.mean([relevance_labels[i] for i in ce_ranking[:3]]):<20.2f}"
      f"{np.mean([relevance_labels[i] for i in be_ranking[:3]]):<20.2f}")

# Calculate NDCG@5 for both
from sklearn.metrics import ndcg_score
ce_ndcg = ndcg_score([relevance_labels], [cross_encoder_scores.tolist()], k=5)
be_ndcg = ndcg_score([relevance_labels], [bi_encoder_scores.tolist()], k=5)
print(f"{'NDCG@5':<30}{ce_ndcg:<20.4f}{be_ndcg:<20.4f}")
print(f"{'Speed advantage':<30}{'-':<20}{ce_latency/be_latency:<20.1f}x faster")

## 5. ColBERT — Late Interaction Re-ranking

### Core Idea

ColBERT (Contextualized Late Interaction over BERT) is a **compromise** between bi-encoders and cross-encoders. It encodes query and document independently (like a bi-encoder) but retains **per-token embeddings** and computes a fine-grained interaction score at query time.

### Architecture

**Step 1: Independent Encoding (offline for documents)**

$$
E_q = \text{BERT}_Q(q) \in \mathbb{R}^{m \times d} \quad \text{(m token embeddings)}
$$

$$
E_d = \text{BERT}_D(d) \in \mathbb{R}^{n \times d} \quad \text{(n token embeddings)}
$$

**Step 2: Late Interaction (MaxSim)**

For each query token, find its maximum similarity with any document token:

$$
s(q, d) = \sum_{i=1}^{m} \max_{j \in [1, n]} \; E_q[i] \cdot E_d[j]^T
$$

This is the **MaxSim** operator. Each query token "soft-matches" to its best-aligned document token.

### Why MaxSim Works

The MaxSim operator captures:
- **Exact matches**: A query token matches its identical counterpart in the document
- **Semantic matches**: A query token matches a semantically similar document token
- **Partial relevance**: Not all query tokens need to match (sum aggregation)

### Complexity Analysis

$$
\text{Encoding}: O(m^2 \cdot H) + O(n^2 \cdot H) \quad \text{(independent, document can be cached)}
$$

$$
\text{Scoring}: O(m \cdot n \cdot d) \quad \text{(much cheaper than cross-attention)}
$$

Compared to Cross-Encoder: $$O((m+n)^2 \cdot H)$$ with $$H \gg d$$

### Storage Trade-off

ColBERT stores per-token embeddings (not just one vector per document):

$$
\text{Storage per doc} = n \times d \times 4 \text{ bytes} \approx 128 \times 128 \times 4 = 64\text{KB}
$$

vs. Bi-Encoder: $$d \times 4 = 768 \times 4 = 3\text{KB}$$

This is a ~20x storage overhead, mitigated by compression (ColBERTv2 uses residual compression).

### Pros
- **Near cross-encoder accuracy**: Token-level interactions capture fine-grained semantics
- **Document embeddings are pre-computable**: Offline indexing like bi-encoders
- **Faster than cross-encoders**: No full transformer forward pass per pair
- **Interpretable**: Can visualize which query tokens matched which document tokens

### Cons
- **High storage cost**: Per-token embeddings (mitigated by compression in ColBERTv2)
- **More complex infrastructure**: Requires specialized indexing (PLAID, late interaction search)
- **Still slower than pure bi-encoder**: MaxSim computation over all token pairs
- **Training complexity**: Requires careful negative sampling and distillation

In [0]:
# ============================================================
# METHOD 3: ColBERT-STYLE LATE INTERACTION (from scratch)
# Implementing MaxSim scoring with per-token embeddings
# ============================================================

class ColBERTReranker:
    """
    ColBERT-style late interaction re-ranker.
    
    Key insight: Encode query and document independently (per-token),
    then compute MaxSim for scoring.
    
    Score(q, d) = sum_i max_j (E_q[i] . E_d[j])
    """
    
    def __init__(self, model_name: str = "bert-base-uncased", dim: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        # ColBERT uses a linear projection to reduce dimensionality
        self.linear = nn.Linear(self.encoder.config.hidden_size, dim)
        self.dim = dim
        
    @torch.no_grad()
    def encode_tokens(self, text: str, max_length: int = 128) -> torch.Tensor:
        """Encode text into per-token embeddings (not pooled!)."""
        inputs = self.tokenizer(
            text, return_tensors="pt", 
            max_length=max_length, truncation=True, padding=True
        )
        outputs = self.encoder(**inputs)
        # Get ALL token embeddings (not just [CLS])
        token_embeddings = outputs.last_hidden_state  # (1, seq_len, hidden_size)
        # Project to lower dimension (ColBERT uses 128d)
        projected = self.linear(token_embeddings)  # (1, seq_len, dim)
        # L2 normalize each token embedding
        normalized = F.normalize(projected, p=2, dim=-1)
        return normalized.squeeze(0)  # (seq_len, dim)
    
    def maxsim_score(self, query_embeddings: torch.Tensor, 
                     doc_embeddings: torch.Tensor) -> float:
        """
        Compute MaxSim: for each query token, find max similarity 
        with any document token, then sum.
        
        Score = sum_i max_j (q_i . d_j)
        """
        # Compute all pairwise similarities: (m, n)
        similarity_matrix = torch.matmul(query_embeddings, doc_embeddings.T)
        
        # For each query token, take the maximum similarity across all doc tokens
        max_similarities = similarity_matrix.max(dim=1).values  # (m,)
        
        # Sum over all query tokens
        return max_similarities.sum().item()
    
    def rerank(self, query: str, documents: List[str]) -> List[Tuple[int, float]]:
        """Re-rank documents using ColBERT MaxSim."""
        self.encoder.eval()
        
        # Encode query tokens
        query_emb = self.encode_tokens(query)  # (m, dim)
        
        scores = []
        for doc in documents:
            # Encode document tokens (in production, this would be pre-computed)
            doc_emb = self.encode_tokens(doc)  # (n, dim)
            score = self.maxsim_score(query_emb, doc_emb)
            scores.append(score)
        
        # Sort by score descending
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked


# Run ColBERT re-ranking
print("Loading ColBERT-style re-ranker...")
colbert = ColBERTReranker("bert-base-uncased", dim=128)

start_time = time.time()
colbert_results = colbert.rerank(query, candidates)
colbert_latency = time.time() - start_time

print(f"\n{'='*70}")
print(f"ColBERT LATE INTERACTION RESULTS")
print(f"{'='*70}")
print(f"Projection dimension: {colbert.dim}")
print(f"Latency: {colbert_latency*1000:.1f}ms for {len(candidates)} candidates")
print(f"\n{'Rank':<6}{'Score':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, (idx, score) in enumerate(colbert_results, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{score:<12.4f}{relevance_labels[idx]:<12}{preview}")

# Visualize the MaxSim alignment for top result
print(f"\n\n--- MaxSim Token Alignment Visualization ---")
query_emb = colbert.encode_tokens(query)
top_idx = colbert_results[0][0]
doc_emb = colbert.encode_tokens(candidates[top_idx])
sim_matrix = torch.matmul(query_emb, doc_emb.T)

query_tokens = colbert.tokenizer.tokenize(query)[:10]
doc_tokens = colbert.tokenizer.tokenize(candidates[top_idx])[:15]

print(f"\nQuery tokens -> Best matching document token:")
for i, qt in enumerate(query_tokens):
    best_j = sim_matrix[i+1, 1:len(doc_tokens)+1].argmax().item()  # +1 to skip [CLS]
    best_sim = sim_matrix[i+1, best_j+1].item()
    print(f"  '{qt}' -> '{doc_tokens[best_j]}' (sim={best_sim:.3f})")

del colbert
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. MonoT5 — Sequence-to-Sequence Re-ranking

### Core Idea

MonoT5 reformulates re-ranking as a **text generation** problem. Given a query-document pair, the model generates either "true" or "false" to indicate relevance. The probability of generating "true" is used as the relevance score.

### Architecture

**Input format:**

$$
\text{Input} = \text{"Query: } q \text{ Document: } d \text{ Relevant:"}
$$

**Scoring via generation probability:**

$$
s(q, d) = P(\text{"true"} \mid \text{Input}) = \frac{e^{z_{\text{true}}}}{e^{z_{\text{true}}} + e^{z_{\text{false}}}}
$$

where $$z_{\text{true}}$$ and $$z_{\text{false}}$$ are the logits for the tokens "true" and "false" respectively.

### Why Text-to-Text Works for Ranking

T5's pre-training on diverse text-to-text tasks gives it:
1. **Strong language understanding** from massive pre-training
2. **Flexible input format**: Can incorporate instructions, context
3. **Transfer learning**: Benefits from T5's multi-task pre-training

### Mathematical Formulation

The encoder-decoder architecture computes:

$$
h_{\text{enc}} = \text{T5-Encoder}(\text{Input})
$$

$$
P(y_t | y_{<t}, h_{\text{enc}}) = \text{softmax}(W_o \cdot \text{T5-Decoder}(y_{<t}, h_{\text{enc}}))
$$

For ranking, we only need the **first generated token's probability**:

$$
s(q, d) = \frac{\exp(\text{logit}_{\text{"true"}})}{\exp(\text{logit}_{\text{"true"}}) + \exp(\text{logit}_{\text{"false"}})}
$$

### Variants

| Model | Parameters | NDCG@10 (MS MARCO) |
| --- | --- | --- |
| MonoT5-small | 60M | 0.368 |
| MonoT5-base | 220M | 0.383 |
| MonoT5-large | 770M | 0.392 |
| MonoT5-3B | 3B | 0.401 |

### Pros
- **Leverages T5's strong pre-training**: Excellent language understanding
- **Flexible prompting**: Can add instructions, few-shot examples
- **Scales well**: Larger models consistently improve
- **Zero-shot capability**: Works reasonably without fine-tuning

### Cons
- **Very slow**: Encoder-decoder is heavier than encoder-only cross-encoders
- **Memory hungry**: T5-3B requires ~12GB+ GPU memory
- **Wasteful computation**: Full decoder pass for a single binary token
- **Harder to distill**: Complex architecture makes knowledge distillation challenging
- **Overkill for simple queries**: Massive model for a binary classification task

In [0]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# ============================================================
# METHOD 4: MonoT5 RE-RANKING
# Using T5 as a pointwise re-ranker
# ============================================================

class MonoT5Reranker:
    """
    MonoT5: Re-ranking via sequence-to-sequence generation.
    
    The model is prompted with "Query: {q} Document: {d} Relevant:"
    and we use P("true") as the relevance score.
    """
    
    def __init__(self, model_name: str = "castorini/monot5-base-msmarco"):
        print(f"Loading MonoT5 model: {model_name}")
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.model.eval()
        
        # Get token IDs for "true" and "false"
        self.true_token_id = self.tokenizer.encode("true", add_special_tokens=False)[0]
        self.false_token_id = self.tokenizer.encode("false", add_special_tokens=False)[0]
    
    def score(self, query: str, document: str) -> float:
        """
        Score a single (query, document) pair.
        
        Returns P("true" | "Query: q Document: d Relevant:")
        """
        # Format input as MonoT5 expects
        input_text = f"Query: {query} Document: {document} Relevant:"
        
        # Tokenize
        inputs = self.tokenizer(
            input_text, return_tensors="pt", 
            max_length=512, truncation=True
        )
        
        # Generate logits for the first output token
        with torch.no_grad():
            # Decoder input: start token
            decoder_input_ids = self.tokenizer(
                "<pad>", return_tensors="pt", add_special_tokens=False
            ).input_ids
            
            outputs = self.model(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                decoder_input_ids=decoder_input_ids
            )
        
        # Extract logits for "true" and "false" tokens
        logits = outputs.logits[0, -1, :]  # Last position logits
        true_logit = logits[self.true_token_id].item()
        false_logit = logits[self.false_token_id].item()
        
        # Softmax over just true/false
        score = np.exp(true_logit) / (np.exp(true_logit) + np.exp(false_logit))
        return score
    
    def rerank(self, query: str, documents: List[str]) -> List[Tuple[int, float]]:
        """Re-rank all documents."""
        scores = [self.score(query, doc) for doc in documents]
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked


# Run MonoT5 re-ranking
monot5 = MonoT5Reranker("castorini/monot5-base-msmarco")

start_time = time.time()
monot5_results = monot5.rerank(query, candidates)
monot5_latency = time.time() - start_time

print(f"\n{'='*70}")
print(f"MonoT5 RE-RANKING RESULTS")
print(f"{'='*70}")
print(f"Model: castorini/monot5-base-msmarco (220M params)")
print(f"Latency: {monot5_latency*1000:.1f}ms for {len(candidates)} candidates")
print(f"Per-candidate: {monot5_latency*1000/len(candidates):.1f}ms")
print(f"\n{'Rank':<6}{'P(true)':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, (idx, score) in enumerate(monot5_results, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{score:<12.4f}{relevance_labels[idx]:<12}{preview}")

del monot5
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 7. LLM-based Listwise Re-ranking

### Core Idea

Instead of scoring documents independently (pointwise), a **listwise re-ranker** considers all candidates simultaneously and directly outputs the optimal permutation. Modern LLMs (GPT-4, Claude, Llama) can perform this as a zero-shot or few-shot task.

### Formal Definition

Given query $$q$$ and candidate set $$\mathcal{D} = \{d_1, \ldots, d_k\}$$, the listwise model outputs a permutation:

$$
\pi^* = \arg\max_{\pi \in S_k} P(\pi \mid q, \mathcal{D})
$$

where $$S_k$$ is the set of all $$k!$$ permutations.

### Approaches

**1. Direct Permutation Generation (RankGPT)**

$$
\text{Prompt} = \text{"Rank these documents by relevance to: } q \text{"}\; + \; \text{enumerate}(\mathcal{D})
$$

$$
\text{Output} = [3, 1, 5, 2, 4, \ldots] \quad \text{(document indices in ranked order)}
$$

**2. Sliding Window (for large candidate sets)**

When $$k$$ exceeds context length, use a sliding window of size $$w$$:

$$
\text{For } i = k-w, k-2w, \ldots, 0: \quad \text{rerank}(\mathcal{D}[i:i+w])
$$

This is a bubble-sort-like approach with $$O(k/w)$$ LLM calls.

**3. Pairwise Tournament**

Compare documents pairwise and aggregate:

$$
s(d_i) = \sum_{j \neq i} \mathbb{1}[\text{LLM prefers } d_i \text{ over } d_j]
$$

Cost: $$O(k^2)$$ comparisons (often reduced via sorting algorithms to $$O(k \log k)$$).

### Theoretical Advantage: Listwise vs. Pointwise

Pointwise scoring can produce **inconsistent rankings** because each score is independent:

$$
\text{Pointwise: } s(q, d_i) \text{ is independent of } s(q, d_j)
$$

Listwise sees all candidates and can make **comparative judgments**:

$$
\text{Listwise: } P(d_i \succ d_j \mid q, \mathcal{D}) \neq P(d_i \succ d_j \mid q, \{d_i, d_j\})
$$

The ranking of $$d_i$$ vs $$d_j$$ can depend on what other documents are present (contrast effect).

### Pros
- **Zero-shot capable**: No training data needed, works out of the box
- **Handles complex reasoning**: Can understand nuance, negation, multi-hop
- **Listwise comparisons**: Considers all candidates jointly
- **Flexible**: Can incorporate instructions, persona, domain knowledge via prompt
- **Explainable**: Can generate reasoning for rankings

### Cons
- **Extremely expensive**: LLM inference costs ($0.01-0.10 per re-ranking call)
- **High latency**: 1-10 seconds per re-ranking call
- **Position bias**: LLMs tend to favor documents at certain positions in the prompt
- **Non-deterministic**: Stochastic generation means rankings can vary
- **Context length limits**: Cannot fit many long documents
- **Hard to fine-tune**: Expensive and unstable to fine-tune for ranking
- **Reliability**: May refuse, hallucinate indices, or produce malformed output

In [0]:
# ============================================================
# METHOD 5: LLM-BASED LISTWISE RE-RANKING
# Simulating the RankGPT approach with a local scoring method
# ============================================================

class LLMListwiseReranker:
    """
    Simulates LLM-based listwise re-ranking (RankGPT-style).
    
    In production, this would call an LLM API (GPT-4, Claude, etc.)
    with a prompt asking it to rank documents.
    
    Here we demonstrate:
    1. The prompt construction
    2. The sliding window approach
    3. A pairwise tournament using a cross-encoder as proxy
    """
    
    def __init__(self, cross_encoder_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        # Using cross-encoder as a proxy for LLM pairwise judgments
        self.scorer = CrossEncoder(cross_encoder_model)
    
    @staticmethod
    def build_rankgpt_prompt(query: str, documents: List[str], 
                             window_size: int = None) -> str:
        """
        Build the RankGPT-style prompt for listwise re-ranking.
        This is what you'd send to GPT-4/Claude in production.
        """
        prompt = f"""I will provide you with {len(documents)} passages, each indicated by a numerical identifier []. 
Rank the passages based on their relevance to the search query: {query}

"""
        for i, doc in enumerate(documents):
            prompt += f"[{i+1}] {doc[:200]}\n\n"
        
        prompt += f"""Search Query: {query}
Rank the {len(documents)} passages above based on their relevance to the search query. 
All the passages should be included and listed using the identifier. 
The output format should be [] > [] > ... (most relevant to least relevant).
The most relevant passages should be listed first.
Output the ranking:"""
        return prompt
    
    def pairwise_tournament(self, query: str, documents: List[str]) -> List[Tuple[int, float]]:
        """
        Pairwise tournament: Compare all pairs and count wins.
        Score(d_i) = number of documents d_i beats.
        
        In production, each comparison would be an LLM call:
        "Which is more relevant to '{query}': [A] or [B]?"
        """
        n = len(documents)
        wins = np.zeros(n)
        
        for i in range(n):
            for j in range(i + 1, n):
                # Score both directions
                score_i = self.scorer.predict([[query, documents[i]]])[0]
                score_j = self.scorer.predict([[query, documents[j]]])[0]
                
                if score_i > score_j:
                    wins[i] += 1
                else:
                    wins[j] += 1
        
        ranked = sorted(enumerate(wins), key=lambda x: x[1], reverse=True)
        return ranked
    
    def sliding_window_rerank(self, query: str, documents: List[str], 
                              window_size: int = 4, step: int = 2) -> List[int]:
        """
        Sliding window re-ranking (for when candidate list exceeds LLM context).
        
        Process: Start from bottom, slide window upward, re-rank within window.
        Like bubble sort - relevant documents "bubble up" to the top.
        """
        n = len(documents)
        current_order = list(range(n))
        
        # Slide window from bottom to top
        for start in range(max(0, n - window_size), -1, -step):
            end = min(start + window_size, n)
            window_indices = current_order[start:end]
            
            # Re-rank within window (in production: LLM call)
            window_docs = [documents[i] for i in window_indices]
            scores = self.scorer.predict([[query, d] for d in window_docs])
            
            # Sort window by score
            sorted_window = [window_indices[i] for i in np.argsort(scores)[::-1]]
            current_order[start:end] = sorted_window
        
        return current_order


# Demonstrate the approach
reranker = LLMListwiseReranker()

# Show the RankGPT prompt (what you'd send to an LLM)
print("=" * 70)
print("EXAMPLE RankGPT PROMPT (sent to LLM in production):")
print("=" * 70)
prompt = LLMListwiseReranker.build_rankgpt_prompt(query, candidates[:4])
print(prompt[:1500] + "\n...")

# Run pairwise tournament
print(f"\n\n{'='*70}")
print("PAIRWISE TOURNAMENT RESULTS")
print(f"{'='*70}")
start_time = time.time()
tournament_results = reranker.pairwise_tournament(query, candidates)
tournament_latency = time.time() - start_time

print(f"Comparisons made: {len(candidates) * (len(candidates)-1) // 2}")
print(f"Latency: {tournament_latency*1000:.1f}ms")
print(f"\n{'Rank':<6}{'Wins':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, (idx, wins) in enumerate(tournament_results, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{int(wins):<12}{relevance_labels[idx]:<12}{preview}")

# Run sliding window
print(f"\n\n{'='*70}")
print("SLIDING WINDOW RE-RANKING RESULTS")
print(f"{'='*70}")
start_time = time.time()
sw_order = reranker.sliding_window_rerank(query, candidates, window_size=4, step=2)
sw_latency = time.time() - start_time

print(f"Window size: 4, Step: 2")
print(f"Latency: {sw_latency*1000:.1f}ms")
print(f"\n{'Rank':<6}{'DocIdx':<12}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, idx in enumerate(sw_order, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{idx:<12}{relevance_labels[idx]:<12}{preview}")

## 8. Reciprocal Rank Fusion (RRF)

### Core Idea

RRF is a **score-free** fusion method that combines rankings from multiple retrieval systems without requiring score normalization. It uses only the **rank position** of each document across different rankers.

### Mathematical Formulation

Given $$R$$ different ranking systems and a constant $$k$$ (typically 60):

$$
\text{RRF}(d) = \sum_{r=1}^{R} \frac{1}{k + \text{rank}_r(d)}
$$

where $$\text{rank}_r(d)$$ is the position of document $$d$$ in the $$r$$-th ranking (1-indexed).

### Why $$k = 60$$?

The constant $$k$$ controls the balance between top-ranked and lower-ranked documents:

- **Small $$k$$** (e.g., 1): Top positions dominate. $$\frac{1}{1+1} = 0.5$$ vs $$\frac{1}{1+10} = 0.09$$
- **Large $$k$$** (e.g., 1000): All positions contribute similarly. $$\frac{1}{1000+1} \approx \frac{1}{1000+10}$$
- **$$k = 60$$**: Empirically found to work well across diverse IR tasks (Cormack et al., 2009)

### Properties

**1. Score-agnostic**: Only uses rank positions, no score normalization needed

$$
\text{No need to align: BM25 scores} \in [0, 30] \text{ with cosine similarity} \in [-1, 1]
$$

**2. Diminishing returns**: The contribution of rank $$r$$ decreases harmonically:

$$
\frac{\partial}{\partial r} \frac{1}{k + r} = -\frac{1}{(k + r)^2} < 0
$$

**3. Robust to outliers**: A single bad ranker cannot dominate because contributions are bounded:

$$
\frac{1}{k + 1} \leq \text{contribution per ranker} \leq \frac{1}{k + N}
$$

### Use Cases in RAG

- Combining **BM25** (lexical) + **Dense Retrieval** (semantic) = Hybrid Search
- Combining **multiple embedding models** (e.g., different domains)
- Combining **chunk-level** + **document-level** retrieval
- Combining **retrieval** + **re-ranker** outputs for ensemble re-ranking

### Pros
- **No training required**: Pure algorithmic fusion
- **Score normalization free**: Works with heterogeneous scoring systems
- **Simple and fast**: $$O(R \cdot k)$$ computation
- **Robust**: Consistently improves over individual rankers
- **Widely adopted**: Default in Elasticsearch, Azure AI Search, many RAG frameworks

### Cons
- **Ignores score magnitude**: A document ranked #1 with score 0.99 and one with 0.51 are treated equally
- **Fixed combination**: No learned weights for different rankers
- **Assumes independent errors**: Works best when rankers make different mistakes
- **Not a true re-ranker**: Fusion method, not a semantic re-scoring model
- **$$k$$ sensitivity**: Performance can vary with $$k$$ choice (though 60 is robust)

In [0]:
# ============================================================
# METHOD 6: RECIPROCAL RANK FUSION (RRF)
# Combining multiple rankers without score normalization
# ============================================================

def reciprocal_rank_fusion(
    rankings: List[List[int]], 
    k: int = 60,
    weights: List[float] = None
) -> List[Tuple[int, float]]:
    """
    Reciprocal Rank Fusion: Combine multiple rankings.
    
    RRF(d) = sum_r weight_r / (k + rank_r(d))
    
    Args:
        rankings: List of rankings, each is a list of document indices ordered by relevance
        k: Constant (default 60, from Cormack et al. 2009)
        weights: Optional weights for each ranker (default: equal)
    
    Returns:
        List of (doc_index, rrf_score) tuples, sorted by RRF score descending
    """
    if weights is None:
        weights = [1.0] * len(rankings)
    
    n_docs = max(max(r) for r in rankings) + 1
    rrf_scores = np.zeros(n_docs)
    
    for ranking, weight in zip(rankings, weights):
        for rank_position, doc_idx in enumerate(ranking, 1):  # 1-indexed rank
            rrf_scores[doc_idx] += weight / (k + rank_position)
    
    # Sort by RRF score
    ranked = sorted(enumerate(rrf_scores), key=lambda x: x[1], reverse=True)
    return ranked


# Generate rankings from different systems
# Ranking 1: Cross-Encoder
ranking_ce = list(ce_ranking)

# Ranking 2: Bi-Encoder (Dense Retrieval)
ranking_be = list(be_ranking)

# Ranking 3: BM25 simulation (using term overlap as proxy)
from collections import Counter
def simple_bm25_proxy(query: str, documents: List[str]) -> List[int]:
    """Simple BM25-like scoring using term frequency."""
    query_terms = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_terms = Counter(doc.lower().split())
        score = sum(doc_terms.get(t, 0) for t in query_terms)
        # IDF-like weighting: rare query terms get higher weight
        score_idf = sum(
            doc_terms.get(t, 0) * np.log(len(documents) / (1 + sum(1 for d in documents if t in d.lower())))
            for t in query_terms
        )
        scores.append(score_idf)
    return list(np.argsort(scores)[::-1])

ranking_bm25 = simple_bm25_proxy(query, candidates)

# Apply RRF
rrf_results = reciprocal_rank_fusion(
    rankings=[ranking_ce, ranking_be, ranking_bm25],
    k=60,
    weights=[1.0, 1.0, 1.0]  # Equal weights
)

print(f"{'='*70}")
print(f"RECIPROCAL RANK FUSION RESULTS")
print(f"{'='*70}")
print(f"Combining: Cross-Encoder + Bi-Encoder + BM25-proxy")
print(f"k = 60 (standard)")
print(f"\nIndividual rankings (top-4):")
print(f"  Cross-Encoder: {ranking_ce[:4]}")
print(f"  Bi-Encoder:    {ranking_be[:4]}")
print(f"  BM25-proxy:    {ranking_bm25[:4]}")
print(f"\n{'Rank':<6}{'RRF Score':<14}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, (idx, score) in enumerate(rrf_results[:len(candidates)], 1):
    if score > 0:
        preview = candidates[idx][:80] + "..."
        print(f"{rank:<6}{score:<14.6f}{relevance_labels[idx]:<12}{preview}")

# Demonstrate effect of k parameter
print(f"\n\n--- Effect of k parameter ---")
print(f"{'k':<8}{'Top-1 Doc':<12}{'Top-3 Docs':<20}{'NDCG@5'}")
print(f"{'-'*55}")
for k_val in [1, 10, 30, 60, 100, 500]:
    rrf_k = reciprocal_rank_fusion([ranking_ce, ranking_be, ranking_bm25], k=k_val)
    rrf_scores_arr = [0.0] * len(candidates)
    for idx, score in rrf_k:
        if idx < len(candidates):
            rrf_scores_arr[idx] = score
    ndcg = ndcg_score([relevance_labels], [rrf_scores_arr], k=5)
    top3 = [idx for idx, _ in rrf_k[:3]]
    print(f"{k_val:<8}{rrf_k[0][0]:<12}{str(top3):<20}{ndcg:.4f}")

## 9. Maximal Marginal Relevance (MMR)

### Core Idea

MMR (Carbonell & Goldstein, 1998) re-ranks by balancing **relevance** to the query with **diversity** among selected documents. It greedily selects documents that are relevant but not redundant with already-selected ones.

### Mathematical Formulation

$$
\text{MMR} = \arg\max_{d_i \in \mathcal{D} \setminus S} \left[ \lambda \cdot \text{Sim}(d_i, q) - (1 - \lambda) \cdot \max_{d_j \in S} \text{Sim}(d_i, d_j) \right]
$$

where:
- $$S$$ is the set of already-selected documents
- $$\mathcal{D} \setminus S$$ is the remaining candidates
- $$\lambda \in [0, 1]$$ balances relevance vs. diversity
- $$\text{Sim}(d_i, q)$$ = relevance of $$d_i$$ to query $$q$$
- $$\text{Sim}(d_i, d_j)$$ = similarity between documents (redundancy)

### Interpretation of $$\lambda$$

| $$\lambda$$ | Behavior |
| --- | --- |
| $$\lambda = 1$$ | Pure relevance ranking (no diversity) |
| $$\lambda = 0$$ | Pure diversity (select most dissimilar documents) |
| $$\lambda = 0.5$$ | Equal balance |
| $$\lambda = 0.7$$ | Typical for RAG (favor relevance, mild diversity) |

### Why MMR Matters for RAG

In RAG, the LLM receives the top-$$m$$ retrieved passages. If these are all paraphrases of the same information:
- **Wasted context window**: Redundant information consumes tokens
- **Missing coverage**: Other relevant aspects are excluded
- **Poor generation quality**: LLM may be overconfident in one aspect

MMR ensures the LLM sees **diverse, complementary** evidence.

### Greedy Algorithm Complexity

Selecting $$m$$ documents from $$k$$ candidates:

$$
\text{Cost} = O(m \cdot k \cdot d) \quad \text{(m iterations, each comparing k-|S| candidates to |S| selected)}
$$

### Pros
- **Reduces redundancy**: Critical for RAG context quality
- **Tunable**: $$\lambda$$ provides direct control over diversity-relevance trade-off
- **Simple and effective**: Greedy algorithm with closed-form scoring
- **Complementary**: Can be applied on top of any other re-ranker
- **Well-suited for RAG**: Maximizes information in limited context window

### Cons
- **Greedy**: Not globally optimal (selecting best at each step ≠ best overall set)
- **Requires pairwise similarity**: $$O(m \cdot k)$$ similarity computations
- **$$\lambda$$ tuning**: Optimal value depends on domain and downstream task
- **Doesn't improve relevance**: Only reshuffles; can push relevant docs down
- **Embedding-dependent diversity**: Diversity is only as good as the embedding space

In [0]:
# ============================================================
# METHOD 7: MAXIMAL MARGINAL RELEVANCE (MMR)
# Balancing relevance with diversity
# ============================================================

def mmr_rerank(
    query_embedding: np.ndarray,
    doc_embeddings: np.ndarray,
    relevance_scores: np.ndarray,
    lambda_param: float = 0.7,
    top_k: int = None
) -> List[Tuple[int, float]]:
    """
    Maximal Marginal Relevance re-ranking.
    
    MMR = argmax_{d_i} [lambda * Sim(d_i, q) - (1-lambda) * max_{d_j in S} Sim(d_i, d_j)]
    
    Args:
        query_embedding: Query vector (d,)
        doc_embeddings: Document vectors (k, d)
        relevance_scores: Pre-computed relevance scores (k,)
        lambda_param: Balance between relevance (1.0) and diversity (0.0)
        top_k: Number of documents to select (default: all)
    
    Returns:
        Ordered list of (doc_index, mmr_score) tuples
    """
    if top_k is None:
        top_k = len(relevance_scores)
    
    n_docs = len(relevance_scores)
    
    # Normalize relevance scores to [0, 1]
    rel_normalized = (relevance_scores - relevance_scores.min()) / \
                     (relevance_scores.max() - relevance_scores.min() + 1e-8)
    
    # Compute pairwise document similarities
    doc_norms = doc_embeddings / (np.linalg.norm(doc_embeddings, axis=1, keepdims=True) + 1e-8)
    doc_similarities = doc_norms @ doc_norms.T  # (k, k)
    
    # Greedy selection
    selected = []
    selected_set = set()
    remaining = set(range(n_docs))
    mmr_scores = []
    
    for _ in range(min(top_k, n_docs)):
        best_idx = -1
        best_mmr = -np.inf
        
        for idx in remaining:
            # Relevance component
            relevance = rel_normalized[idx]
            
            # Diversity component: max similarity to any already-selected doc
            if selected:
                max_sim_to_selected = max(doc_similarities[idx][s] for s in selected)
            else:
                max_sim_to_selected = 0.0
            
            # MMR score
            mmr = lambda_param * relevance - (1 - lambda_param) * max_sim_to_selected
            
            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = idx
        
        selected.append(best_idx)
        selected_set.add(best_idx)
        remaining.remove(best_idx)
        mmr_scores.append((best_idx, best_mmr))
    
    return mmr_scores


# Compute embeddings for MMR
query_emb_np = bi_encoder.encode(query, convert_to_numpy=True)
doc_embs_np = bi_encoder.encode(candidates, convert_to_numpy=True)

# Use cross-encoder scores as relevance (best of both worlds)
print(f"{'='*70}")
print(f"MAXIMAL MARGINAL RELEVANCE (MMR) RESULTS")
print(f"{'='*70}")

# Compare different lambda values
for lam in [1.0, 0.7, 0.5, 0.3]:
    mmr_results = mmr_rerank(
        query_embedding=query_emb_np,
        doc_embeddings=doc_embs_np,
        relevance_scores=cross_encoder_scores,
        lambda_param=lam,
        top_k=len(candidates)
    )
    
    print(f"\n--- Lambda = {lam} {'(pure relevance)' if lam == 1.0 else '(balanced)' if lam == 0.5 else ''} ---")
    print(f"{'Rank':<6}{'MMR Score':<14}{'Relevance':<12}{'Document Preview'}")
    print(f"{'-'*70}")
    for rank, (idx, score) in enumerate(mmr_results[:5], 1):
        preview = candidates[idx][:70] + "..."
        print(f"{rank:<6}{score:<14.4f}{relevance_labels[idx]:<12}{preview}")

# Show diversity effect
print(f"\n\n--- Diversity Analysis ---")
print("Pairwise cosine similarities between top-3 selections:")
for lam in [1.0, 0.5]:
    mmr_results = mmr_rerank(query_emb_np, doc_embs_np, cross_encoder_scores, lam)
    top3_idx = [idx for idx, _ in mmr_results[:3]]
    top3_embs = doc_embs_np[top3_idx]
    top3_norms = top3_embs / (np.linalg.norm(top3_embs, axis=1, keepdims=True) + 1e-8)
    sims = top3_norms @ top3_norms.T
    avg_sim = (sims.sum() - 3) / 6  # Exclude diagonal
    print(f"  Lambda={lam}: Avg pairwise similarity = {avg_sim:.4f} (lower = more diverse)")

## 10. Learning to Rank (LTR) — LambdaMART

### Core Idea

Learning to Rank uses **handcrafted or learned features** from (query, document) pairs and trains a ranking model (typically gradient-boosted trees) to predict relevance. LambdaMART is the most successful LTR algorithm, combining **LambdaRank** gradients with **MART** (Multiple Additive Regression Trees).

### Feature Engineering

LTR models use rich feature vectors $$\phi(q, d) \in \mathbb{R}^F$$:

$$
\phi(q, d) = [\text{BM25}(q,d), \; \cos(e_q, e_d), \; \text{CE}(q,d), \; \text{len}(d), \; \text{freshness}(d), \; \ldots]
$$

Common features:
- Lexical: BM25, TF-IDF, query term coverage
- Semantic: Bi-encoder similarity, cross-encoder score
- Document quality: Length, freshness, authority, click-through rate
- Query-document: Overlap ratios, entity matches, title match

### LambdaMART Objective

LambdaMART directly optimizes NDCG via **lambda gradients**. For a pair of documents $$(d_i, d_j)$$ where $$y_i > y_j$$ ($$d_i$$ is more relevant):

$$
\lambda_{ij} = \frac{-\sigma}{1 + e^{\sigma(s_i - s_j)}} \cdot |\Delta \text{NDCG}_{ij}|
$$

where:
- $$s_i, s_j$$ are model scores
- $$\sigma$$ is a sigmoid parameter
- $$|\Delta \text{NDCG}_{ij}|$$ is the change in NDCG if positions of $$d_i$$ and $$d_j$$ are swapped

The key insight: the gradient is **weighted by the metric change** from swapping the pair. Swapping documents near the top matters more.

### NDCG (Normalized Discounted Cumulative Gain)

$$
\text{DCG}@k = \sum_{i=1}^{k} \frac{2^{y_i} - 1}{\log_2(i + 1)}
$$

$$
\text{NDCG}@k = \frac{\text{DCG}@k}{\text{IDCG}@k}
$$

where $$\text{IDCG}@k$$ is the DCG of the ideal ranking.

### The $$\Delta\text{NDCG}$$ Swap Weight

$$
|\Delta \text{NDCG}_{ij}| = \left| \frac{2^{y_i} - 2^{y_j}}{\text{IDCG}} \cdot \left( \frac{1}{\log_2(\text{rank}_i + 1)} - \frac{1}{\log_2(\text{rank}_j + 1)} \right) \right|
$$

This makes the algorithm focus on correcting mistakes that have the biggest impact on ranking quality.

### Pros
- **Feature-rich**: Can incorporate any signal (behavioral, content, metadata)
- **Fast inference**: Tree ensemble inference is $$O(F \cdot \text{depth} \cdot \text{n\_trees})$$, typically <1ms
- **Directly optimizes NDCG**: Lambda gradients approximate the true metric gradient
- **Interpretable**: Feature importances, tree visualization, SHAP values
- **Production-proven**: Used at Google, Bing, LinkedIn, Airbnb for decades
- **Handles heterogeneous features**: Numerical, categorical, embedding-derived

### Cons
- **Feature engineering burden**: Requires significant domain expertise and iteration
- **No end-to-end learning**: Features are fixed; model cannot discover new representations
- **Training data requirements**: Needs graded relevance judgments (expensive to collect)
- **Cold start**: New documents without features are hard to rank
- **Staleness**: Features may become stale (e.g., click-through rates change)
- **Less effective for semantic matching**: Trees can't capture complex token interactions like transformers

In [0]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

# ============================================================
# METHOD 8: LEARNING TO RANK (LambdaMART)
# Feature-based ranking with gradient boosted trees
# ============================================================

class LambdaMARTReranker:
    """
    LambdaMART re-ranker using LightGBM.
    
    Key components:
    1. Feature engineering: Extract (query, document) features
    2. LambdaRank objective: Directly optimizes NDCG
    3. Gradient boosted trees: Fast and interpretable
    """
    
    def __init__(self):
        self.model = None
        self.feature_names = [
            "bm25_score", "cosine_similarity", "query_term_coverage",
            "doc_length", "title_overlap", "exact_match_ratio",
            "query_doc_len_ratio", "unique_term_overlap"
        ]
    
    def extract_features(self, query: str, document: str, 
                         query_emb: np.ndarray = None, 
                         doc_emb: np.ndarray = None) -> np.ndarray:
        """
        Extract handcrafted features for a (query, document) pair.
        In production, you'd have 100+ features including click data.
        """
        query_terms = set(query.lower().split())
        doc_terms = document.lower().split()
        doc_term_set = set(doc_terms)
        
        # Feature 1: BM25-like score (simplified)
        tf = sum(1 for t in doc_terms if t in query_terms)
        bm25_score = tf / (tf + 1.2 * (1 - 0.75 + 0.75 * len(doc_terms) / 100))
        
        # Feature 2: Cosine similarity (if embeddings provided)
        if query_emb is not None and doc_emb is not None:
            cosine_sim = np.dot(query_emb, doc_emb) / (
                np.linalg.norm(query_emb) * np.linalg.norm(doc_emb) + 1e-8
            )
        else:
            cosine_sim = 0.0
        
        # Feature 3: Query term coverage
        coverage = len(query_terms & doc_term_set) / len(query_terms)
        
        # Feature 4: Document length
        doc_length = len(doc_terms)
        
        # Feature 5: Title overlap (first sentence as proxy)
        first_sentence = document.split('.')[0].lower().split()
        title_overlap = len(query_terms & set(first_sentence)) / len(query_terms)
        
        # Feature 6: Exact match ratio
        exact_matches = sum(1 for t in query_terms if t in doc_term_set)
        exact_match_ratio = exact_matches / len(query_terms)
        
        # Feature 7: Query-document length ratio
        len_ratio = len(query.split()) / (len(doc_terms) + 1)
        
        # Feature 8: Unique term overlap
        unique_overlap = len(query_terms & doc_term_set) / (len(query_terms | doc_term_set) + 1)
        
        return np.array([
            bm25_score, cosine_sim, coverage, doc_length,
            title_overlap, exact_match_ratio, len_ratio, unique_overlap
        ])
    
    def train(self, queries: List[str], docs_per_query: List[List[str]], 
              labels_per_query: List[List[int]], 
              query_embs: np.ndarray = None, doc_embs_list: List[np.ndarray] = None):
        """
        Train LambdaMART model.
        
        Args:
            queries: List of queries
            docs_per_query: List of document lists (one per query)
            labels_per_query: List of relevance label lists
            query_embs: Pre-computed query embeddings
            doc_embs_list: Pre-computed document embeddings per query
        """
        all_features = []
        all_labels = []
        group_sizes = []
        
        for i, (q, docs, labels) in enumerate(zip(queries, docs_per_query, labels_per_query)):
            q_emb = query_embs[i] if query_embs is not None else None
            for j, (doc, label) in enumerate(zip(docs, labels)):
                d_emb = doc_embs_list[i][j] if doc_embs_list is not None else None
                features = self.extract_features(q, doc, q_emb, d_emb)
                all_features.append(features)
                all_labels.append(label)
            group_sizes.append(len(docs))
        
        X = np.array(all_features)
        y = np.array(all_labels)
        
        # Train LightGBM with LambdaRank objective
        train_data = lgb.Dataset(
            X, label=y, group=group_sizes,
            feature_name=self.feature_names
        )
        
        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'ndcg_eval_at': [3, 5, 10],
            'num_leaves': 31,
            'learning_rate': 0.1,
            'n_estimators': 100,
            'verbose': -1
        }
        
        self.model = lgb.train(
            params, train_data, 
            num_boost_round=100,
            valid_sets=[train_data],
            callbacks=[lgb.log_evaluation(period=0)]  # Suppress output
        )
    
    def predict(self, query: str, documents: List[str],
                query_emb: np.ndarray = None, 
                doc_embs: np.ndarray = None) -> List[Tuple[int, float]]:
        """Score and rank documents."""
        features = []
        for j, doc in enumerate(documents):
            d_emb = doc_embs[j] if doc_embs is not None else None
            features.append(self.extract_features(query, doc, query_emb, d_emb))
        
        X = np.array(features)
        scores = self.model.predict(X)
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked


# Create synthetic training data (in production, you'd use click logs or human annotations)
# We'll create a few additional queries for training
training_queries = [
    "What are the benefits of transformer architecture over RNNs?",
    "How does attention mechanism work in neural networks?",
    "What is transfer learning in NLP?",
    "Explain the difference between BERT and GPT models",
]

# Use the same candidates for simplicity (in production, each query has its own candidate set)
training_docs = [candidates] * len(training_queries)
training_labels = [
    relevance_labels,
    [2, 1, 2, 0, 1, 1, 3, 0],  # For attention query
    [1, 0, 3, 0, 1, 2, 1, 0],  # For transfer learning query
    [1, 0, 2, 0, 0, 3, 1, 0],  # For BERT vs GPT query
]

# Compute embeddings for all training data
all_query_embs = bi_encoder.encode(training_queries, convert_to_numpy=True)
all_doc_embs = [bi_encoder.encode(docs, convert_to_numpy=True) for docs in training_docs]

# Train LambdaMART
ltr = LambdaMARTReranker()
ltr.train(
    training_queries, training_docs, training_labels,
    query_embs=all_query_embs, doc_embs_list=all_doc_embs
)

# Predict on our test query
start_time = time.time()
ltr_results = ltr.predict(query, candidates, query_emb_np, doc_embs_np)
ltr_latency = time.time() - start_time

print(f"{'='*70}")
print(f"LEARNING TO RANK (LambdaMART) RESULTS")
print(f"{'='*70}")
print(f"Features: {ltr.feature_names}")
print(f"Trees: {ltr.model.num_trees()}")
print(f"Latency: {ltr_latency*1000:.2f}ms for {len(candidates)} candidates")
print(f"\n{'Rank':<6}{'LTR Score':<14}{'Relevance':<12}{'Document Preview'}")
print(f"{'-'*70}")
for rank, (idx, score) in enumerate(ltr_results, 1):
    preview = candidates[idx][:80] + "..."
    print(f"{rank:<6}{score:<14.4f}{relevance_labels[idx]:<12}{preview}")

# Feature importance
print(f"\n\n--- Feature Importance ---")
importances = ltr.model.feature_importance(importance_type='gain')
for name, imp in sorted(zip(ltr.feature_names, importances), key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp / max(importances) * 30)
    print(f"  {name:<25} {imp:>8.1f} {bar}")

## 11. Comprehensive Comparison

### Performance vs. Latency Trade-off

| Method | Accuracy (NDCG@10) | Latency (100 docs) | Memory | Training Data | Complexity |
| --- | --- | --- | --- | --- | --- |
| Cross-Encoder | ★★★★★ (highest) | 100–500ms (GPU) | ~1GB | Moderate | Low |
| ColBERT | ★★★★ | 20–80ms | ~2GB + index | Moderate | Medium |
| MonoT5 | ★★★★★ (T5-3B) | 500–2000ms | 6–12GB | Moderate | Medium |
| LLM Listwise | ★★★★★ (GPT-4) | 2–10s | API cost | None (zero-shot) | Low |
| Bi-Encoder | ★★★ | <5ms | ~500MB | Large | Low |
| RRF | ★★★★ | <1ms | Negligible | None | Very Low |
| MMR | ★★★ (diversity) | <5ms | Negligible | None | Very Low |
| LambdaMART | ★★★★ | <1ms | ~10MB | Large + features | High (features) |

### Decision Framework

```
Latency budget > 500ms?
├─ YES → MonoT5-3B or LLM Listwise (highest accuracy)
└─ NO
    ├─ Latency < 100ms?
    │   ├─ YES → ColBERT or RRF ensemble
    │   └─ Can tolerate 100-500ms? → Cross-Encoder (best ROI)
    └─ Need diversity?
        ├─ YES → Apply MMR on top of any re-ranker
        └─ NO → Cross-Encoder or LambdaMART
```

### Production Architecture Patterns

**Pattern 1: Two-Stage (Most Common)**
$$
\text{BM25/Dense} \xrightarrow{k=100} \text{Cross-Encoder} \xrightarrow{m=5} \text{LLM}
$$

**Pattern 2: Three-Stage (High-Volume)**
$$
\text{BM25} \xrightarrow{k=1000} \text{ColBERT} \xrightarrow{k=100} \text{Cross-Encoder} \xrightarrow{m=5} \text{LLM}
$$

**Pattern 3: Ensemble + Diversity**
$$
\text{RRF}(\text{BM25}, \text{Dense}) \xrightarrow{k=50} \text{Cross-Encoder} \xrightarrow{} \text{MMR}(\lambda=0.7) \xrightarrow{m=5} \text{LLM}
$$

**Pattern 4: Feature-Rich (E-commerce/Web Search)**
$$
\text{Dense} \xrightarrow{k=200} \text{LambdaMART}(\text{features} + \text{CE\_score}) \xrightarrow{m=10} \text{User}
$$

In [0]:
# ============================================================
# FINAL COMPARISON: All Re-ranking Methods Side by Side
# ============================================================

print(f"{'='*80}")
print(f"FINAL COMPARISON: ALL RE-RANKING METHODS")
print(f"{'='*80}")
print(f"\nQuery: '{query}'")
print(f"Candidates: {len(candidates)} documents")
print(f"Ground truth relevant docs (label >= 2): indices {[i for i, l in enumerate(relevance_labels) if l >= 2]}")

# Collect all rankings
all_methods = {
    "Cross-Encoder": (list(ce_ranking), cross_encoder_scores.tolist(), ce_latency),
    "Bi-Encoder": (list(be_ranking), bi_encoder_scores.tolist(), be_latency),
    "ColBERT (MaxSim)": ([idx for idx, _ in colbert_results], [s for _, s in colbert_results], colbert_latency),
    "RRF (CE+BE+BM25)": (
        [idx for idx, _ in rrf_results[:len(candidates)]], 
        [s for _, s in rrf_results[:len(candidates)]], 
        0.001  # Negligible
    ),
    "LambdaMART": ([idx for idx, _ in ltr_results], [s for _, s in ltr_results], ltr_latency),
}

# Add MonoT5 if it ran successfully
try:
    all_methods["MonoT5"] = (
        [idx for idx, _ in monot5_results], 
        [s for _, s in monot5_results], 
        monot5_latency
    )
except:
    pass

# Print comparison table
print(f"\n{'Method':<22}{'Top-1':<8}{'Top-3 (indices)':<20}{'NDCG@5':<10}{'Latency (ms)':<15}")
print(f"{'-'*75}")

for method, (ranking, scores, latency) in all_methods.items():
    top1_relevance = relevance_labels[ranking[0]]
    top3_indices = ranking[:3]
    
    # Calculate NDCG@5
    scores_full = [0.0] * len(candidates)
    for i, idx in enumerate(ranking):
        if idx < len(candidates):
            scores_full[idx] = len(ranking) - i  # Convert rank to score
    ndcg = ndcg_score([relevance_labels], [scores_full], k=5)
    
    print(f"{method:<22}{top1_relevance:<8}{str(top3_indices):<20}{ndcg:<10.4f}{latency*1000:<15.1f}")

# Ranking agreement analysis
print(f"\n\n{'='*80}")
print(f"RANKING AGREEMENT ANALYSIS")
print(f"{'='*80}")
print("\nKendall's Tau correlation between methods (top-5 positions):")
from scipy.stats import kendalltau

method_names = list(all_methods.keys())
print(f"\n{'':20}", end="")
for name in method_names:
    print(f"{name[:10]:>12}", end="")
print()

for i, name_i in enumerate(method_names):
    print(f"{name_i[:18]:<20}", end="")
    for j, name_j in enumerate(method_names):
        rank_i = all_methods[name_i][0][:5]
        rank_j = all_methods[name_j][0][:5]
        tau, _ = kendalltau(rank_i, rank_j)
        print(f"{tau:>12.3f}", end="")
    print()

## 12. Production Deployment Considerations

### Latency Budget Allocation

For a typical RAG system with a **500ms total SLA**:

$$
T_{\text{total}} = T_{\text{retrieval}} + T_{\text{rerank}} + T_{\text{LLM}} \leq 500\text{ms}
$$

$$
\approx 50\text{ms} + 100\text{ms} + 300\text{ms} = 450\text{ms} \quad \text{(with buffer)}
$$

### Batching and Parallelism

**Cross-Encoder batching** (GPU):
$$
\text{Throughput} = \frac{\text{batch\_size}}{T_{\text{forward\_pass}}} \approx \frac{32}{50\text{ms}} = 640 \text{ scorings/sec}
$$

**Document-parallel ColBERT** (documents are independent):
$$
\text{Throughput} = \frac{k}{T_{\text{MaxSim}}} \approx \frac{100}{5\text{ms}} = 20{,}000 \text{ scorings/sec}
$$

### Model Distillation Pipeline

For production, distill expensive models into cheaper ones:

$$
\text{Teacher (Cross-Encoder)} \xrightarrow{\text{distill}} \text{Student (Bi-Encoder or ColBERT)}
$$

$$
\mathcal{L}_{\text{distill}} = \text{KL}\left( \frac{e^{s^T_i / \tau}}{\sum_j e^{s^T_j / \tau}} \;\Big\|\; \frac{e^{s^S_i / \tau}}{\sum_j e^{s^S_j / \tau}} \right)
$$

where $$s^T$$ and $$s^S$$ are teacher and student scores, and $$\tau$$ is the distillation temperature.

### Key Takeaways

1. **Cross-Encoder is the default choice** for most RAG systems (best accuracy/complexity ratio)
2. **RRF is free accuracy** — always combine sparse + dense before re-ranking
3. **MMR is essential for RAG** — diversity in context directly improves generation
4. **ColBERT when latency is critical** — near cross-encoder accuracy at 5–10x speed
5. **LambdaMART when you have features** — behavioral signals (clicks) are gold
6. **LLM re-ranking for low-volume, high-stakes** — legal, medical, enterprise search
7. **Always measure on YOUR data** — benchmark numbers don't transfer perfectly

### Recommended Reading

- Nogueira et al. (2020): *"Passage Re-ranking with BERT"* — Cross-Encoder fundamentals
- Khattab & Zaharia (2020): *"ColBERT: Efficient and Effective Passage Search"*
- Nogueira et al. (2020): *"Document Ranking with a Pretrained Sequence-to-Sequence Model"* — MonoT5
- Sun et al. (2023): *"Is ChatGPT Good at Search? Investigating LLMs as Re-Ranking Agents"* — RankGPT
- Cormack et al. (2009): *"Reciprocal Rank Fusion"*
- Burges (2010): *"From RankNet to LambdaRank to LambdaMART"*